In [1]:
import torch
from transformers import ( 
    AutoTokenizer,  
    AutoModelForSequenceClassification,  
    pipeline,
    BitsAndBytesConfig
) 
from peft import PeftModel 

d:\dev\workspace\ai\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
base_model_id = "beomi/kcbert-base"  
qlora_model_path = "./saved_models/qlora_sentiment/final_model" 

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForSequenceClassification.from_pretrained( 
    base_model_id, 
    num_labels=2,
    #quantization_config=bnb_config,
    device_map="auto"
) 

tokenizer = AutoTokenizer.from_pretrained(qlora_model_path) 
qlora_model = PeftModel.from_pretrained(base_model, qlora_model_path) 

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1148.89it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [6]:
pipe = pipeline( 
    task="text-classification", 
    model=qlora_model, 
    tokenizer=tokenizer
) 

In [7]:
texts = [ 
    # 긍정 데이터 
    "전체적인 분위기가 좋아서 편하게 볼 수 있었어요.", 
    "스토리는 평범했지만 연출 덕분에 재미있었어요.", 
    "배우들의 연기가 자연스러워서 몰입이 잘 됐어요.", 
    "큰 기대 없이 봤는데 생각보다 괜찮았어요.", 
    "잔잔하지만 끝나고 나서 여운이 남는 영화였어요.", 
     
    # 부정 데이터 
    "이야기가 늘어져서 중간부터 집중이 안 됐어요.", 
    "연출이 과해서 오히려 몰입을 방해했어요.", 
    "캐릭터 행동이 이해되지 않아서 답답했어요.", 
    "분위기는 잡으려는 것 같은데 내용이 부족했어요.", 
    "전체적으로 뭔가 아쉬운 느낌이 많이 남았어요." 
] 

predicts = pipe(texts) 

In [8]:
print(predicts)

[{'label': 'LABEL_0', 'score': 0.5341537594795227}, {'label': 'LABEL_1', 'score': 0.7024308443069458}, {'label': 'LABEL_1', 'score': 0.6510328650474548}, {'label': 'LABEL_1', 'score': 0.5431060194969177}, {'label': 'LABEL_1', 'score': 0.6263733506202698}, {'label': 'LABEL_1', 'score': 0.5655139088630676}, {'label': 'LABEL_1', 'score': 0.6660687923431396}, {'label': 'LABEL_1', 'score': 0.6923714876174927}, {'label': 'LABEL_1', 'score': 0.6242761015892029}, {'label': 'LABEL_1', 'score': 0.6510384678840637}]


In [9]:
for text, pred in zip(texts, predicts): 
    if pred['label'] == 'LABEL_1': 
        result = "긍정" 
    else: 
        result = "부정" 
    print(f'입력 데이터: "{text}", 감성 분석 결과: {result}')

입력 데이터: "전체적인 분위기가 좋아서 편하게 볼 수 있었어요.", 감성 분석 결과: 부정
입력 데이터: "스토리는 평범했지만 연출 덕분에 재미있었어요.", 감성 분석 결과: 긍정
입력 데이터: "배우들의 연기가 자연스러워서 몰입이 잘 됐어요.", 감성 분석 결과: 긍정
입력 데이터: "큰 기대 없이 봤는데 생각보다 괜찮았어요.", 감성 분석 결과: 긍정
입력 데이터: "잔잔하지만 끝나고 나서 여운이 남는 영화였어요.", 감성 분석 결과: 긍정
입력 데이터: "이야기가 늘어져서 중간부터 집중이 안 됐어요.", 감성 분석 결과: 긍정
입력 데이터: "연출이 과해서 오히려 몰입을 방해했어요.", 감성 분석 결과: 긍정
입력 데이터: "캐릭터 행동이 이해되지 않아서 답답했어요.", 감성 분석 결과: 긍정
입력 데이터: "분위기는 잡으려는 것 같은데 내용이 부족했어요.", 감성 분석 결과: 긍정
입력 데이터: "전체적으로 뭔가 아쉬운 느낌이 많이 남았어요.", 감성 분석 결과: 긍정
